 ### ETAPA DE IMPORTAÇÃO

In [1]:
from pathlib import Path
import sys
import zipfile

sys.path.insert(0, str(Path.cwd().resolve().parent))

from _MySQL import banco, config

from scripts import extrair
from scripts import transformar

### Etapa de Download da Base do Drive

In [2]:
extrair.baixar_base_drive()

Iniciando o Download...


Downloading...
From (original): https://drive.google.com/uc?id=1Sru-TYSYo-cn-L9WW2DUwhyIUdkM3fIe
From (redirected): https://drive.google.com/uc?id=1Sru-TYSYo-cn-L9WW2DUwhyIUdkM3fIe&confirm=t&uuid=a7ea210a-d8cc-4e27-a41c-93ea2ffa923e
To: /home/calliari/Documents/GitHub/Projeto_final_SCTEC/data/viagens_2025_6meses.zip
100%|██████████| 67.7M/67.7M [00:06<00:00, 10.1MB/s]


### Etapa da Ingestão dos dados para o raw de cada tabela

In [4]:
try:
    conexao = banco.conectar()
    caminho_zip = extrair.localizar_zip()

    with zipfile.ZipFile(caminho_zip) as zip_aberto:
        for arquivo in config.ARQUIVOS.values():
            extrair.carregar_csv(conexao, zip_aberto, arquivo["csv"], arquivo["tabela_raw"])

    conexao.close()
    print("=== Camada RAW concluida com sucesso! ===")
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise

      Carregando raw_viagem ...
      -> 341860 linhas em raw_viagem
      Carregando raw_pagamento ...
      -> 606916 linhas em raw_pagamento
      Carregando raw_passagem ...
      -> 167260 linhas em raw_passagem
      Carregando raw_trecho ...
      -> 763349 linhas em raw_trecho
=== Camada RAW concluida com sucesso! ===


### ETAPA DE TRANSFORMAÇÃO

In [5]:
## Limpeza das tabelas silver antes de popular

lista_silver = [
    "silver_pagamento",
    "silver_passagem",
    "silver_trecho",
    "silver_viagem"
]

conexao = banco.conectar()
try:
    transformar.limpar_silver(conexao,lista_silver)

    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise


Iniciando a limpeza das tabelas Silver...
Limpeza Concluída
Foram limpas o total de 4 tabelas


In [6]:
### Popular tabela silver_viagem
sql = """
INSERT INTO silver_viagem
(
    id_viagem,
    num_proposta,
    situacao,
    viagem_urgente,
    cod_orgao_superior,
    nome_orgao_superior,
    nome_viajante,
    cargo,
    data_inicio,
    data_fim,
    destinos,
    motivo,
    valor_diarias,
    valor_passagens,
    valor_devolucao,
    valor_outros_gastos
)
SELECT
    UPPER(r.id_viagem),
    UPPER(r.num_proposta),
    UPPER(r.situacao),
    UPPER(r.viagem_urgente),
    UPPER(r.cod_orgao_superior),
    UPPER(r.nome_orgao_superior),
    UPPER(r.nome_viajante),
    UPPER(r.cargo),
	STR_TO_DATE(NULLIF(TRIM(r.data_inicio),  ''), '%d/%m/%Y'),
    STR_TO_DATE(NULLIF(TRIM(r.data_fim),  ''), '%d/%m/%Y'),
	UPPER(r.destinos),
	UPPER(r.motivo),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(r.valor_diarias),    ''), '.', ''), ',', '.') AS DECIMAL(10,2)),
    CAST(REPLACE(REPLACE(NULLIF(TRIM( r.valor_passagens), ''), '.', ''), ',', '.') AS DECIMAL(10,2)),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(r.valor_devolucao), ''), '.', ''), ',', '.') AS DECIMAL(10,2)),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(r.valor_outros_gastos), ''), '.', ''), ',', '.') AS DECIMAL(10,2))

FROM raw_viagem r;
"""
try:
    print ("Iniciando a etapa de Popular tabela silver_viagem...")
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise


### Popular coluna valor_total da tabela silver_viagem

sql = """
    UPDATE silver_viagem
    SET
	valor_total = valor_diarias + valor_passagens + valor_devolucao + valor_outros_gastos ;
    """

try:
    print ("Iniciando a etapa de Popular coluna valor_total da tabela silver_viagem...")
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise


### Popular coluna duracao_dias da tabela silver_viagem

sql = """
    UPDATE silver_viagem 
    SET 
    duracao_dias = DATEDIFF(data_fim, data_inicio);
    """

try:
    print ("Iniciando a etapa de Popular coluna duracao_dias da tabela silver_viagem...")
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise




Iniciando a etapa de Popular tabela silver_viagem...
Iniciando a etapa de Popular coluna valor_total da tabela silver_viagem...
Iniciando a etapa de Popular coluna duracao_dias da tabela silver_viagem...


In [7]:
### Popular tabela silver_passagem

sql = """ 
INSERT INTO silver_passagem(
	id_viagem,
    meio_transporte,
	pais_origem_ida,
    uf_origem_ida,
    cidade_origem_ida,
    pais_destino_ida,
    uf_destino_ida,
    cidade_destino_ida,
    valor_passagem,
    taxa_servico,
    data_emissao
)
SELECT
	UPPER(sv.id_viagem),
    UPPER(r.meio_transporte),
	UPPER(r.pais_origem_ida),
    UPPER(r.uf_origem_ida),
    UPPER(r.cidade_origem_ida),
    UPPER(r.pais_destino_ida),
    UPPER(r.uf_destino_ida),
    UPPER(r.cidade_destino_ida),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(r.valor_passagem),    ''), '.', ''), ',', '.') AS DECIMAL(10,2)),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(r.taxa_servico),    ''), '.', ''), ',', '.') AS DECIMAL(10,2)),
    STR_TO_DATE(NULLIF(TRIM(r.data_emissao), ''), '%d/%m/%Y')
    
FROM raw_passagem r
INNER JOIN silver_viagem sv
    ON r.id_viagem = sv.id_viagem;

"""
try:
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise

In [8]:
## POPULAR TABELA silver_trecho
sql = """ 
INSERT INTO silver_trecho(
	id_viagem,
	sequencia_trecho,
    origem_data,
    origem_uf,
    origem_cidade,
    destino_data,
    destino_uf,
    destino_cidade,
    meio_transporte,
    numero_diarias
)
SELECT
	UPPER(sv.id_viagem),
	rt.sequencia_trecho,
    STR_TO_DATE(NULLIF(TRIM(rt.origem_data),  ''), '%d/%m/%Y'),
    UPPER(rt.origem_uf),
    UPPER(rt.origem_cidade),
    STR_TO_DATE(NULLIF(TRIM(rt.destino_data),  ''), '%d/%m/%Y'),
    UPPER(rt.destino_uf),
    UPPER(rt.destino_cidade),
    UPPER(rt.meio_transporte),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(rt.numero_diarias),    ''), '.', ''), ',', '.') AS DECIMAL(10,2))
    FROM 
        raw_trecho rt
    INNER JOIN 
        silver_viagem sv
    ON rt.id_viagem = sv.id_viagem;

"""
try:
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise

In [1]:
## POPULAR silver_pagamento
sql = """ 
INSERT INTO silver_pagamento (
	id_viagem,
    num_proposta,
    nome_orgao_pagador,
    nome_ug_pagadora,
    tipo_pagamento,
    valor
)
SELECT
	UPPER(sv.id_viagem),
	UPPER(rp.num_proposta),
    UPPER(rp.nome_orgao_pagador),
    UPPER(rp.nome_ug_pagadora),
    UPPER(rp.tipo_pagamento),
    CAST(REPLACE(REPLACE(NULLIF(TRIM(rp.valor),    ''), '.', ''), ',', '.') AS DECIMAL(10,2))
    
FROM raw_pagamento rp
INNER JOIN silver_viagem sv 
	on rp.id_viagem = sv.id_viagem
"""
try:
    conexao = banco.conectar()
    banco.executar(conexao,sql)
    conexao.close()
except Exception as erro:
    print("[ERRO] Algo deu errado:", erro)
    raise


[ERRO] Algo deu errado: name 'banco' is not defined


NameError: name 'banco' is not defined